# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata
# Display the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets and fields by their @id
record_sets = list(dataset.record_sets())  # Each is a RecordSet object
print("Record sets available:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# Show fields for each record set and their @id
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' (@id: {rs.id}):")
    for f in rs.fields:
        print(f"  - {f.name} (@id: {f.id}, dataType: {f.data_type})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract from all record sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Extract records (returns list of dicts)
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet '@id': {record_set_id} columns:")
    print(df.columns.tolist())
    print(df.head())

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0]
df_main = dataframes[main_record_set_id]
print(f"\nFields in main record set (@id: {main_record_set_id}): {df_main.columns.tolist()}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Example: Analyze GAD-7 scores (anxiety assessment)
# Field @id may be something like 'gad7_score' -- check from field overview above
# Here, we use the first numeric field found
numeric_fields = [col for col in df_main.columns if df_main[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = 10
    filtered_df = df_main[df_main[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalizing numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a demographic field
    group_fields = [col for col in df_main.columns if (
        df_main[col].dtype == 'object' and col != numeric_field)]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found in main record set. EDA skipped.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of the main numeric field (e.g., GAD-7 score)
if numeric_fields:
    plt.hist(df_main[numeric_field].dropna(), bins=15, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field} in Kilifi Survey Records")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped analysis is available
    if group_fields:
        grouped_df.plot(kind='bar', y=f"mean_{numeric_field}", legend=False)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore the Kilifi County FAIR² mental health survey dataset using the `mlcroissant` library. 
- We loaded and reviewed metadata, explored the structure and fields using their `@id`s, extracted records, and performed EDA including filtering and normalization.
- Visualizations allowed us to inspect the distribution of key numeric scores and their groupings by demographic categories.
- These steps make the dataset ready for AI/ML pipelines and for public health analysis.

You can extend this workflow to perform more advanced analyses and visualizations using the standardized Croissant schema and `mlcroissant` tools.